# Прототип AI-ассистента персонального сайта (этап 3)

**Тема:** персональный сайт с AI-ассистентом для представления профессиональных компетенций.
**Автор:** Андрис Янчевскис.

Ассистент отвечает на вопросы посетителей (рекрутеров, работодателей, заказчиков) об опыте Андриса
только по внешней базе знаний (RAG). Исследовательский вопрос: даёт ли **редактируемая база знаний (RAG A)**
более точные и честные ответы, чем база, полученная **парсингом страниц сайта (RAG B1)** или
**страниц и PDF-файлов сайта (RAG B2)**. Код ассистента одинаков для всех баз, меняется только база.

## Схема работы

```
вопрос
  │
  ▼
[LLM-вызов 1] detect_intent ──► намерение + поисковый запрос
  │                              (off_topic → вежливый отказ без поиска)
  ▼
[RAG] embed_texts + search ──► top_k фрагментов базы знаний (FAISS)
  │                              (нет фрагментов выше порога → отказ)
  ▼
[LLM-вызов 2] generate_answer ─► ответ только по найденным фрагментам,
                                 номера источников, признак достаточности
```

## Как запустить

1. Откройте ноутбук в Google Colab. Базы знаний и промпты скачиваются автоматически по ссылке
   на архив `stage3_prototype_data.zip` и распаковываются в `/content/stage3_prototype`.
   В локальном Jupyter запустите ноутбук из папки, где лежат файлы `records_*.jsonl` и папка `prompts`.
2. В Google Colab добавьте секрет `OPENAI_API_KEY` (значок ключа на левой панели) и разрешите
   ноутбуку доступ к нему. В локальном Jupyter задайте переменную окружения `OPENAI_API_KEY`.
3. Выполните ячейки по порядку. В последней ячейке запускается `main()`: введите вопрос и базу
   знаний — `A`, `B1`, `B2` или `ALL` для сравнения.
4. Индексы FAISS строятся при первом запуске и сохраняются в папку `index/`; повторно эмбеддинги
   не создаются.

Ключ в ноутбуке не хранится. Ответы без ключа не подставляются: при ошибке API выводится
честное сообщение об ошибке.

## 1. Подготовка окружения

Установка библиотек. Файлы баз знаний и промпты в Google Colab скачиваются автоматически
в следующей ячейке по ссылке на архив; подключать Google Drive не нужно.

In [ ]:
%pip install -q faiss-cpu openai pydantic numpy gdown

## 2. Импорт библиотек и пути к файлам

Все пути — параметры функций со значениями по умолчанию относительно папки ноутбука.

In [ ]:
import json
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Literal

import faiss
import numpy as np
from pydantic import BaseModel, Field, ValidationError

DATA_ZIP_URL = "https://drive.google.com/file/d/1WyBsx_Xeo_NqXfM5TJHUfAQxdYYpdHhV/view?usp=sharing"  # общедоступная ссылка Google Drive на архив с данными
COLAB_DATA_DIR = Path("/content/stage3_prototype")


def download_data(zip_url: str = DATA_ZIP_URL, target_dir: Path = COLAB_DATA_DIR) -> Path:
    """Скачивает архив с базами знаний и промптами по ссылке и распаковывает его в target_dir.

    Если данные уже распакованы в этой сессии Colab, повторно архив не скачивается.
    """
    import zipfile

    if (target_dir / "records_rag_a.jsonl").exists():
        return target_dir
    if not zip_url:
        raise ValueError("Не задана ссылка DATA_ZIP_URL на архив stage3_prototype_data.zip")
    import gdown  # установлен в Google Colab; скачивает файлы Google Drive по ссылке «доступ по ссылке»

    target_dir.mkdir(parents=True, exist_ok=True)
    archive = target_dir.parent / "stage3_prototype_data.zip"
    if not gdown.download(zip_url, str(archive), quiet=True, fuzzy=True):
        raise RuntimeError(f"Не удалось скачать архив: {zip_url}. Проверьте, что у файла включён доступ по ссылке.")
    with zipfile.ZipFile(archive) as zip_file:
        zip_file.extractall(target_dir)
    return target_dir


def data_dir_for_environment(zip_url: str = DATA_ZIP_URL, colab_dir: Path = COLAB_DATA_DIR) -> Path:
    """Папка с файлами ноутбука: в Google Colab — распакованный архив, локально — текущая папка."""
    try:
        import google.colab  # noqa: F401  (модуль есть только в Google Colab)
    except ImportError:
        return Path(".")
    return download_data(zip_url, colab_dir)


DATA_DIR = data_dir_for_environment()  # папка, в которой лежат базы знаний и промпты
print(f"Папка с файлами ноутбука: {DATA_DIR.resolve()}")
DEFAULT_RECORDS_PATH = DATA_DIR / "records_rag_a.jsonl"
DEFAULT_PROMPTS_DIR = DATA_DIR / "prompts"
DEFAULT_INDEX_DIR = DATA_DIR / "index" / "rag_a"

## 3. Настройки и структуры данных

`AssistantConfig` собирает все параметры, которые меняются в экспериментах. Pydantic-модели проверяют ответы LLM на каждом шаге.

In [ ]:
@dataclass(frozen=True)
class AssistantConfig:
    """Все параметры, которые меняются в экспериментах, собраны в одном месте."""

    language: str = "ru"                  # язык промптов и служебных сообщений
    site: str = "ru"                      # версия сайта: фильтр поля site у записей
    chunk_size: int = 800                 # максимальная длина фрагмента, символов
    chunk_overlap: int = 100              # перекрытие соседних фрагментов, символов
    top_k: int = 5                        # сколько фрагментов передавать во второй вызов
    min_score: float = 0.25               # порог косинусного сходства для фрагмента
    embedding_model: str = "text-embedding-3-small"
    llm_model: str = "gpt-4o-mini"
    intent_temperature: float = 0.0
    answer_temperature: float = 0.2
    intent_prompt_version: str = "v1"
    answer_prompt_version: str = "v1"


class IntentResult(BaseModel):
    """Результат первого вызова LLM."""

    intent: Literal["fact", "project", "job_match", "work_terms", "contact", "off_topic"]
    search_query: str = ""
    language: str = "ru"


class AnswerResult(BaseModel):
    """Результат второго вызова LLM."""

    answer: str = Field(min_length=1)
    sufficient_context: bool
    used_sources: list[int] = []


@dataclass
class AssistantResponse:
    """То, что получает интерфейс (ноутбук или сайт)."""

    answer: str
    status: Literal["answered", "insufficient_context", "no_context", "off_topic", "contact"]
    sources: list[dict[str, Any]] = field(default_factory=list)
    log: dict[str, Any] = field(default_factory=dict)


class AssistantError(Exception):
    """Ошибка провайдера LLM или невалидный ответ модели."""

## 4. База знаний и чанкинг

Загрузка записей (для RAG A — только `verified` и `public`), деление текста на фрагменты с перекрытием.

In [ ]:
def load_records(path: str | Path = DEFAULT_RECORDS_PATH, site: str = "ru",
                 require_verified: bool = True) -> list[dict[str, Any]]:
    """Читает records.jsonl и оставляет записи, допущенные к индексации.

    Допускаются записи с public=true, у которых поле site равно "all" или коду
    нужной версии сайта. При require_verified=True (база RAG A) дополнительно
    требуется verified=true. Корпус RAG B, полученный парсингом сайта, вручную
    не проверялся, поэтому загружается с require_verified=False.
    """
    records = []
    with open(path, encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue
            record = json.loads(line)
            if not record.get("public"):
                continue
            if require_verified and not record.get("verified"):
                continue
            if record.get("site", "all") not in ("all", site):
                continue
            records.append(record)
    return records


def split_text(text: str, chunk_size: int = 800, chunk_overlap: int = 100) -> list[str]:
    """Делит текст на фрагменты не длиннее chunk_size символов.

    Граница фрагмента по возможности ставится в конце предложения; следующий
    фрагмент начинается на chunk_overlap символов раньше конца предыдущего.
    """
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap должен быть меньше chunk_size")
    text = " ".join(text.split())
    if len(text) <= chunk_size:
        return [text] if text else []

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            sentence_end = max(text.rfind(". ", start, end), text.rfind("! ", start, end), text.rfind("? ", start, end))
            if sentence_end > start + chunk_size // 2:
                end = sentence_end + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(end - chunk_overlap, start + 1)
    return chunks


def chunk_records(records: list[dict[str, Any]], chunk_size: int = 800, chunk_overlap: int = 100) -> list[dict[str, Any]]:
    """Превращает записи базы в фрагменты для индексации.

    Заголовок записи добавляется в начало каждого фрагмента, чтобы короткий
    фрагмент сохранял смысл. Метаданные источника сохраняются в каждом фрагменте.
    """
    chunks = []
    for record in records:
        parts = split_text(record["content"], chunk_size, chunk_overlap)
        for number, part in enumerate(parts, start=1):
            chunks.append({
                "chunk_id": f"{record['record_id']}#{number}",
                "record_id": record["record_id"],
                "entity_type": record.get("entity_type", ""),
                "title": record.get("title", ""),
                "source_url": record.get("source_url", ""),
                "text": f"{record.get('title', '')}. {part}",
            })
    return chunks

## 5. Эмбеддинги и векторный индекс FAISS

Эмбеддинги OpenAI, L2-нормализация и индекс `IndexFlatIP` (точный поиск по косинусному сходству).

In [ ]:
def embed_texts(texts: list[str], client: Any, model: str = "text-embedding-3-small", batch_size: int = 100) -> np.ndarray:
    """Получает эмбеддинги текстов от провайдера пакетами по batch_size."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        try:
            response = client.embeddings.create(input=batch, model=model)
        except Exception as exc:  # ошибка сети, ключа, баланса и т. п.
            raise AssistantError(f"Ошибка при получении эмбеддингов: {exc}") from exc
        vectors.extend(item.embedding for item in response.data)
    return np.asarray(vectors, dtype=np.float32)


def normalize(vectors: np.ndarray) -> np.ndarray:
    """L2-нормализация: после неё скалярное произведение равно косинусному сходству."""
    vectors = np.asarray(vectors, dtype=np.float32)
    if vectors.ndim == 1:
        vectors = vectors.reshape(1, -1)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return vectors / norms


def build_index(vectors: np.ndarray) -> faiss.Index:
    """Создаёт индекс FAISS IndexFlatIP (точный поиск по косинусному сходству)."""
    vectors = normalize(vectors)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index


def search(query_vector: np.ndarray, index: faiss.Index, chunks: list[dict[str, Any]],
           top_k: int = 5, min_score: float = 0.25) -> list[dict[str, Any]]:
    """Возвращает до top_k фрагментов с косинусным сходством не ниже min_score."""
    if index.ntotal == 0:
        return []
    scores, ids = index.search(normalize(query_vector), min(top_k, index.ntotal))
    hits = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0 or float(score) < min_score:
            continue
        hit = dict(chunks[idx])
        hit["score"] = round(float(score), 4)
        hits.append(hit)
    return hits


def save_index(index: faiss.Index, chunks: list[dict[str, Any]], config: AssistantConfig,
               directory: str | Path = DEFAULT_INDEX_DIR) -> Path:
    """Сохраняет индекс, фрагменты и параметры, с которыми индекс построен."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, str(directory / "index.faiss"))
    with open(directory / "chunks.json", "w", encoding="utf-8") as file:
        json.dump({"config": asdict(config), "chunks": chunks}, file, ensure_ascii=False, indent=2)
    return directory


def load_index(directory: str | Path = DEFAULT_INDEX_DIR) -> tuple[faiss.Index, list[dict[str, Any]], dict[str, Any]]:
    """Загружает индекс, фрагменты и параметры построения."""
    directory = Path(directory)
    index = faiss.read_index(str(directory / "index.faiss"))
    with open(directory / "chunks.json", encoding="utf-8") as file:
        data = json.load(file)
    return index, data["chunks"], data["config"]


def build_knowledge_index(client: Any, config: AssistantConfig,
                          records_path: str | Path = DEFAULT_RECORDS_PATH,
                          require_verified: bool = True) -> tuple[faiss.Index, list[dict[str, Any]]]:
    """Полный цикл индексации: загрузка записей → фрагменты → эмбеддинги → индекс."""
    records = load_records(records_path, site=config.site, require_verified=require_verified)
    chunks = chunk_records(records, config.chunk_size, config.chunk_overlap)
    vectors = embed_texts([chunk["text"] for chunk in chunks], client, config.embedding_model)
    return build_index(vectors), chunks

## 6. Промпты и два обращения к LLM

Промпт 1 определяет намерение и формирует поисковый запрос, промпт 2 отвечает только по найденному контексту. Тексты промптов лежат в папке `prompts/` и имеют версии.

In [ ]:
def load_prompts(config: AssistantConfig, prompts_dir: str | Path = DEFAULT_PROMPTS_DIR) -> dict[str, Any]:
    """Читает версии промптов и служебные сообщения для языка из config."""
    folder = Path(prompts_dir) / config.language
    with open(folder / "messages.json", encoding="utf-8") as file:
        messages = json.load(file)
    return {
        "intent": (folder / f"intent_{config.intent_prompt_version}.txt").read_text(encoding="utf-8"),
        "answer": (folder / f"answer_{config.answer_prompt_version}.txt").read_text(encoding="utf-8"),
        "messages": messages,
    }


def call_llm_json(client: Any, model: str, system_prompt: str, user_content: str, temperature: float) -> dict[str, Any]:
    """Один вызов chat completions с ответом в формате JSON."""
    try:
        completion = client.chat.completions.create(
            model=model,
            temperature=temperature,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ],
        )
        raw = completion.choices[0].message.content or ""
    except Exception as exc:
        raise AssistantError(f"Ошибка обращения к LLM: {exc}") from exc
    try:
        return json.loads(raw)
    except json.JSONDecodeError as exc:
        raise AssistantError(f"LLM вернула невалидный JSON: {raw[:200]}") from exc


def detect_intent(question: str, client: Any, model: str, prompt: str, temperature: float = 0.0) -> IntentResult:
    """LLM-вызов 1: определяет намерение и формирует поисковый запрос."""
    data = call_llm_json(client, model, prompt, f"Вопрос посетителя:\n{question}", temperature)
    try:
        return IntentResult.model_validate(data)
    except ValidationError as exc:
        raise AssistantError(f"Ответ первого шага не соответствует схеме: {data}") from exc


def format_context(hits: list[dict[str, Any]]) -> str:
    """Нумерует найденные фрагменты для второго промпта: [1] ..., [2] ..."""
    return "\n\n".join(f"[{number}] {hit['text']}" for number, hit in enumerate(hits, start=1))


def generate_answer(question: str, hits: list[dict[str, Any]], client: Any, model: str,
                    prompt: str, temperature: float = 0.2) -> AnswerResult:
    """LLM-вызов 2: формирует ответ строго по найденным фрагментам."""
    user_content = f"Контекст:\n{format_context(hits)}\n\nВопрос посетителя:\n{question}"
    data = call_llm_json(client, model, prompt, user_content, temperature)
    try:
        result = AnswerResult.model_validate(data)
    except ValidationError as exc:
        raise AssistantError(f"Ответ второго шага не соответствует схеме: {data}") from exc
    # Оставляем только номера, которые действительно есть в контексте.
    result.used_sources = sorted({n for n in result.used_sources if 1 <= n <= len(hits)})
    return result

## 7. Обработка одного вопроса

`run_assistant` выполняет шаги по схеме и возвращает ответ, статус, источники и журнал шагов.

In [ ]:
def run_assistant(question: str, client: Any, index: faiss.Index, chunks: list[dict[str, Any]],
                  config: AssistantConfig, prompts: dict[str, Any]) -> AssistantResponse:
    """Обрабатывает один вопрос: намерение → поиск → ответ. Возвращает ответ и журнал шагов."""
    started = time.perf_counter()
    question = question.strip()
    log: dict[str, Any] = {"question": question, "config": asdict(config)}
    messages = prompts["messages"]

    intent = detect_intent(question, client, config.llm_model, prompts["intent"], config.intent_temperature)
    log["intent"] = intent.model_dump()

    if intent.intent == "off_topic":
        return AssistantResponse(messages["off_topic"], "off_topic", log=_finish(log, started))
    if intent.intent == "contact" and not intent.search_query.strip():
        return AssistantResponse(messages["contact"], "contact", log=_finish(log, started))

    query = intent.search_query.strip() or question
    query_vector = embed_texts([query], client, config.embedding_model)
    hits = search(query_vector, index, chunks, config.top_k, config.min_score)
    log["hits"] = [{"chunk_id": h["chunk_id"], "score": h["score"]} for h in hits]

    if not hits:
        return AssistantResponse(messages["no_context"], "no_context", log=_finish(log, started))

    result = generate_answer(question, hits, client, config.llm_model, prompts["answer"], config.answer_temperature)
    log["answer"] = result.model_dump()

    sources = [
        {"number": n, "title": hits[n - 1]["title"], "source_url": hits[n - 1]["source_url"],
         "record_id": hits[n - 1]["record_id"]}
        for n in result.used_sources
    ]
    status = "answered" if result.sufficient_context else "insufficient_context"
    return AssistantResponse(result.answer, status, sources, log=_finish(log, started))


def _finish(log: dict[str, Any], started: float) -> dict[str, Any]:
    """Возвращает копию журнала с длительностью обработки вопроса."""
    return {**log, "duration_sec": round(time.perf_counter() - started, 3)}

## 8. Функции ноутбука и тестовый запуск

`ask` задаёт вопрос с выбранными базами, `main` — главная функция: берёт ключ API из секретов Colab
(`OPENAI_API_KEY`), запрашивает вопрос и базу знаний и выводит ответы. Для сравнения баз введите `ALL`.

In [ ]:
CORPORA = {
    "A": {"file": "records_rag_a.jsonl", "require_verified": True, "title": "RAG A — редактируемая база знаний"},
    "B1": {"file": "records_rag_b1.jsonl", "require_verified": False, "title": "RAG B1 — парсинг страниц сайта"},
    "B2": {"file": "records_rag_b2.jsonl", "require_verified": False, "title": "RAG B2 — страницы и PDF сайта"},
}


def index_dir_for(corpus: str, config: AssistantConfig, data_dir: str | Path = DATA_DIR) -> Path:
    """Папка индекса: своя для каждой базы, модели эмбеддингов и параметров чанкинга."""
    name = f"{corpus}_{config.embedding_model}_{config.chunk_size}_{config.chunk_overlap}"
    return Path(data_dir) / "index" / name


def get_corpus_index(corpus: str, client: Any, config: AssistantConfig, corpora: dict = CORPORA,
                     data_dir: str | Path = DATA_DIR) -> tuple[Any, list[dict[str, Any]]]:
    """Загружает сохранённый индекс базы или строит и сохраняет новый."""
    directory = index_dir_for(corpus, config, data_dir)
    if (directory / "index.faiss").exists():
        index, chunks, _ = load_index(directory)
        return index, chunks
    settings = corpora[corpus]
    index, chunks = build_knowledge_index(client, config, Path(data_dir) / settings["file"],
                                          require_verified=settings["require_verified"])
    save_index(index, chunks, config, directory)
    return index, chunks


def ask(question: str, corpus_names: list[str], client: Any, config: AssistantConfig, prompts: dict[str, Any],
        corpora: dict = CORPORA, data_dir: str | Path = DATA_DIR) -> dict[str, AssistantResponse | str]:
    """Задаёт один вопрос ассистенту с каждой из выбранных баз знаний."""
    results: dict[str, AssistantResponse | str] = {}
    for corpus in corpus_names:
        try:
            index, chunks = get_corpus_index(corpus, client, config, corpora, data_dir)
            results[corpus] = run_assistant(question, client, index, chunks, config, prompts)
        except AssistantError as exc:
            results[corpus] = f"Ошибка: {exc}"
    return results


def format_result(corpus: str, result: AssistantResponse | str, corpora: dict = CORPORA) -> str:
    """Готовит понятный текст результата для вывода."""
    lines = [f"=== {corpora[corpus]['title']} ==="]
    if isinstance(result, str):
        return "\n".join(lines + [result])
    lines.append(f"Статус: {result.status}")
    lines.append(result.answer)
    hits = result.log.get("hits", [])
    if hits:
        cited = {source["number"]: source for source in result.sources}
        lines.append("  Фрагменты, переданные модели (✓ — модель сослалась на него в ответе):")
        for number, hit in enumerate(hits, start=1):
            mark = "✓" if number in cited else " "
            title = f" — {cited[number]['title']} ({cited[number]['source_url']})" if number in cited else ""
            lines.append(f"   {mark} [{number}] {hit['chunk_id']} (сходство {hit['score']}){title}")
    return "\n".join(lines)


def read_api_key(secret_name: str = "OPENAI_API_KEY") -> str:
    """Читает ключ OpenAI API из секретов Google Colab, вне Colab — из переменной окружения."""
    import os

    try:
        from google.colab import userdata  # доступно только в Google Colab
        key = userdata.get(secret_name)
    except ImportError:
        key = os.environ.get(secret_name)
    if not key:
        raise RuntimeError(
            f"Ключ не найден. В Colab добавьте секрет {secret_name} (значок ключа слева) и разрешите "
            f"ноутбуку доступ к нему; вне Colab задайте переменную окружения {secret_name}."
        )
    return key


def create_client(api_key: str | None = None) -> Any:
    """Создаёт клиента OpenAI; без параметра ключ читается функцией read_api_key."""
    from openai import OpenAI

    return OpenAI(api_key=api_key or read_api_key())


def main(data_dir: str | Path = DATA_DIR, config: AssistantConfig | None = None, corpora: dict = CORPORA) -> None:
    """Тестовая функция: вопросы вводятся с клавиатуры, ответы выводятся по выбранным базам."""
    config = config or AssistantConfig()
    client = create_client()
    prompts = load_prompts(config, Path(data_dir) / "prompts")
    print("Пустой вопрос завершает работу.")
    while True:
        question = input("\nВопрос: ").strip()
        if not question:
            break
        choice = (input(f"База знаний ({', '.join(corpora)} или ALL) [ALL]: ").strip().upper() or "ALL")
        names = list(corpora) if choice == "ALL" else [choice]
        if any(name not in corpora for name in names):
            print(f"Неизвестная база: {choice}")
            continue
        for corpus, result in ask(question, names, client, config, prompts, corpora, data_dir).items():
            print()
            print(format_result(corpus, result, corpora))

## 9. Тестовая ячейка

Запустите и введите свой вопрос.

In [ ]:
main()